# 04 — Modulating risk factors (H3, exploratory)

**Notebook id**: `04_risk_factors` → vault `03.5-facteurs-risque.md`

Within the **Cyclops** group, which intrinsic factors modulate the **patellofemoral** progression Δ`lesion_pf`? (H3 is **exploratory**.)

- Continuous (age, **BMI** — `imc`, now usable, derived from taille/poids): Spearman ρ + BCa CI.
- Binary (female, smoker, physical work): Mann–Whitney + Cliff δ.
- Multilevel pivot (0/1/2): Kruskal–Wallis + ε².
- All p-values **BH-FDR** corrected (family F2). `inter_surgery_d` is **excluded** — it is time-at-risk / a mediator (H4 outcome), not a risk factor.
- **Sport/occupation sensitivity**: the group→worsened-PF effect **with and without** the Pivot / Physical-work covariates, via the now **Firth-based** `tf.sensitivity_covariate` (stable under the 1/19 separation).

In [ ]:
import sys
from pathlib import Path

current = Path().absolute().parent
sys.path.insert(0, (current / "src").as_posix())


In [ ]:
# --- Setup (idempotent, fresh-kernel reproducible) ---
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

from constants import (
    RANDOM_SEED, SITES, SITES_PF, SITES_FT, SITES_BINARY, SITES_ORDINAL,
    GROUPS, BLOCKS, N_TOTAL, N_TOTAL_ANALYSABLE, N_MENISCUS, N_CYCLOPS,
    SCORE_MAX, SCORE_MAX_COLLAPSED,
)
import loaders
import preprocessing as pp
import tests_freq as tf
import reporting as rpt
import bayes_models as bm
import viz

np.random.seed(RANDOM_SEED)
viz.set_pub_style()


In [ ]:
# --- Load & preprocess (canonical pipeline) ---
df = loaders.load_combined()
df = pp.apply_date_hygiene(df)      # composite-key (group, anonyme) date hygiene
df = pp.add_derived(df)            # lesion_pf/ft, female, deltas, worsened_pf, ...
wide = pp.to_wide(df)             # one row per patient (group, anonyme)
patient = pp.to_patient(df)       # static covariates per patient

# Patient-level covariates joined onto the wide outcomes (for H3 / sensitivity).
_cov = [c for c in ['group','anonyme','female','sexe','pivot_pivot_contact',
                    'travail_physique','tabac','age_at_trauma','imc','taille','poids']
        if c in patient.columns]
merged = wide.merge(patient[_cov], on=['group','anonyme'], how='left')

print('long:', df.shape, '| wide:', wide.shape, '| patient:', patient.shape,
      '| merged:', merged.shape)
# Composite-key sentinel: 19 Anonyme ids are reused across the two sheets.
assert (df.groupby(['group','anonyme']).size() == 2).all(), 'composite key broken'


## 1. Cyclops subset (joined to covariates)

In [ ]:
cyc = merged[merged['group']=='cyclops'].copy()
print('Cyclops subset:', cyc.shape)
print(cyc[['delta_lesion_pf','age_at_trauma','imc','female','tabac',
           'travail_physique','pivot_pivot_contact']].head())


## 2. H3 factor battery vs Δ`lesion_pf` (BH-FDR, exploratory)

In [ ]:
h3 = tf.h3_risk_factors(
    cyc, outcome_col='delta_lesion_pf',
    continuous=('age_at_trauma','imc'),
    binary=('female','tabac','travail_physique'),
    multilevel=('pivot_pivot_contact',),
    q=0.10, n_boot=2000, seed=RANDOM_SEED)
print(h3.to_string(index=False))
print()
print('All H3 associations are exploratory (hypothesis-generating).')


## 3. Kruskal–Wallis detail on `pivot_pivot_contact` (0/1/2)

In [ ]:
if 'pivot_pivot_contact' in cyc.columns:
    sub = cyc[['pivot_pivot_contact','delta_lesion_pf']].apply(
        pd.to_numeric, errors='coerce').dropna()
    sub['pivot_pivot_contact'] = sub['pivot_pivot_contact'].astype(int).astype(str)
    res = tf.kw_dunn(sub, 'pivot_pivot_contact', 'delta_lesion_pf')
    print(f"H={res['statistic']:.3f}  p={res['pvalue']:.3f}  "
          f"eps^2={res['epsilon_sq']:.3f}  n={res['n']}")
    print('Dunn post-hoc (Bonferroni):')
    print(res['posthoc'])


## 4. Sport / occupation sensitivity — group effect WITH vs WITHOUT covariates

**Firth** penalised logistic of `worsened_pf` on group: crude (WITHOUT) vs adjusted (WITH) Pivot + Physical work. Firth is used because the 1/19 meniscus events separate the plain ML logit (ghost OR); `or_*_ci` are the penalised CIs. If the adjusted OR stays comparable to the crude OR, the PF effect is **not** an artefact of the sport/occupation imbalance.

In [ ]:
sens = tf.sensitivity_covariate(
    merged, outcome_col='worsened_pf',
    covariates=('pivot_pivot_contact','travail_physique'))
print('Sport/occupation sensitivity (worsened_pf ~ group), Firth-based:')
print(f"  method        : {sens['method']}  (separation_ml={sens['separation_ml']})")
print(f"  crude (WITHOUT): OR = {sens['or_crude']:.2f}  CI {tuple(round(x,2) for x in sens['or_crude_ci'])}"
      f"  p = {sens['p_crude']:.3f}  n = {sens['n_crude']}")
print(f"  adj.  (WITH)   : OR = {sens['or_adjusted']:.2f}  CI {tuple(round(x,2) for x in sens['or_adjusted_ci'])}"
      f"  p = {sens['p_adjusted']:.3f}  n = {sens['n_adjusted']}")
print(f"  covariates     : {sens['covariates']}")
print(f"  adjusted_ok    : {sens['adjusted_ok']}")


## 5. Spearman correlogram (Cyclops, exploratory)

In [ ]:
cols = [c for c in ['age_at_trauma','imc','taille','poids','inter_surgery_d',
                    'delta_lesion_pf','delta_lesion_ft'] if c in cyc.columns]
fig = viz.correlogram(cyc, cols)
fig


## Sanity asserts

In [ ]:
assert len(cyc) == N_CYCLOPS  # 49 cyclops patients in the subset
assert 'p_adj_bh' in h3.columns
print('Risk-factors asserts passed.')
